# 成交数据 重构

## 元数据
- 原始路径: g:\\我的云端硬盘\\workspace\\projects\\github\\work_station\\代码库\\模版\\成交数据
- 创建时间: 2026-02-15
- 任务ID: T39

## 1. 项目概述

### 1.1 功能描述
成交数据项目专注于收集、处理和分析债券市场的成交数据，包括成交量、成交金额、成交价格、成交结构等，为市场流动性分析、交易策略制定和投资决策提供数据支持。

### 1.2 数据源
- 评级狗API (ratingdog.cn)
  - 登录API: https://auth.ratingdog.cn/api/TokenAuth/Authenticate
  - 成交API: https://host.ratingdog.cn/api/services/app/Bond/QueryTradedHistoricalOfFrontDeskTenants

### 1.3 输出结果
- yq.cjqb表: 全部成交数据
- yq.cj表: 中介成交数据

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 导入标准库
import os
import sys
import json
import time
import random
from datetime import datetime, timedelta
from time import sleep

# 导入第三方库
import pandas as pd
import numpy as np
import requests
import sqlalchemy
from sqlalchemy.sql import text
import pymysql

# 导入配置
from config import (
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR,
    get_db_connection_string,
    RATINGDOG_LOGIN_URL, RATINGDOG_TRADE_URL, RATINGDOG_AGENT_URL,
    RATINGDOG_USERNAME, RATINGDOG_PASSWORD, RATINGDOG_TENANT_ID,
    REQUEST_TIMEOUT, MAX_RESULT_COUNT, MAX_RETRIES,
    get_login_headers, get_api_headers, get_login_data,
    TRADE_DATA_CONFIG
)

print("依赖导入完成")
print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

### 2.2 配置参数

In [ ]:
# 显示当前配置
print("=== 成交数据配置 ===")
print(f"请求超时时间: {REQUEST_TIMEOUT}秒")
print(f"每页最大结果数: {MAX_RESULT_COUNT}")
print(f"最大重试次数: {MAX_RETRIES}")
print(f"评级狗Tenant ID: {RATINGDOG_TENANT_ID}")
print(f"\n流动性阈值配置:")
for k, v in TRADE_DATA_CONFIG['liquidity_thresholds'].items():
    print(f"  {k}: {v}")

## 3. 数据获取

### 3.1 数据源连接

In [ ]:
# 创建数据库连接
def create_db_connection():
    """创建数据库连接"""
    try:
        engine = sqlalchemy.create_engine(
            get_db_connection_string(),
            poolclass=sqlalchemy.pool.NullPool,
            pool_recycle=3600
        )
        connection = engine.connect()
        print("数据库连接成功")
        return engine, connection
    except Exception as e:
        print(f"数据库连接失败: {e}")
        return None, None

# 创建连接
db_engine, db_connection = create_db_connection()

### 3.2 数据读取 - 查询日期范围

In [ ]:
def get_pending_dates(connection, target_table='yq.cjqb'):
    """
    获取待采集的日期范围
    
    Args:
        connection: 数据库连接
        target_table: 目标表名
    
    Returns:
        日期列表
    """
    query = f"""
    SELECT DISTINCT dt 
    FROM bond.marketinfo_abs 
    WHERE dt NOT IN (SELECT DISTINCT dt FROM {target_table})
    ORDER BY dt
    """
    
    try:
        date_list = pd.read_sql(query, connection)
        if len(date_list) > 0:
            print(f"找到 {len(date_list)} 个待采集日期")
            print(f"日期范围: {date_list['dt'].min()} 至 {date_list['dt'].max()}")
        else:
            print("没有待采集的日期")
        return date_list
    except Exception as e:
        print(f"查询日期失败: {e}")
        return pd.DataFrame()

# 获取待采集日期
pending_dates = get_pending_dates(db_connection)
pending_dates.head() if len(pending_dates) > 0 else print("无待采集日期")

## 4. 数据处理

### 4.1 数据清洗

In [ ]:
def clean_trade_data(df):
    """
    清洗成交数据
    
    Args:
        df: 原始成交数据DataFrame
    
    Returns:
        清洗后的DataFrame
    """
    if df is None or len(df) == 0:
        return df
    
    original_count = len(df)
    
    # 处理缺失值
    df = df.where(pd.notna(df), None)
    
    # 统一时间格式（如果存在日期列）
    if 'tradedDate' in df.columns:
        df['tradedDate'] = pd.to_datetime(df['tradedDate'], errors='coerce')
    
    cleaned_count = len(df)
    print(f"数据清洗完成: 原始{original_count}条 -> 清洗后{cleaned_count}条")
    
    return df

print("数据清洗函数定义完成")

### 4.2 数据转换

In [ ]:
def aggregate_trade_data(df, group_by='tradedDate'):
    """
    聚合成交数据
    
    Args:
        df: 成交数据DataFrame
        group_by: 聚合维度
    
    Returns:
        聚合后的DataFrame
    """
    if df is None or len(df) == 0:
        return None
    
    agg_dict = {}
    
    # 根据可用列定义聚合规则
    if 'tradedAmount' in df.columns:
        agg_dict['tradedAmount'] = 'sum'
    if 'tradedPrice' in df.columns:
        agg_dict['tradedPrice'] = 'mean'
    if 'tradedYield' in df.columns:
        agg_dict['tradedYield'] = 'mean'
    
    if group_by in df.columns and agg_dict:
        agg_dict['count'] = ('tradedAmount', 'count') if 'tradedAmount' in df.columns else None
        agg_df = df.groupby(group_by).agg(agg_dict).reset_index()
        return agg_df
    
    return df

print("数据聚合函数定义完成")

## 5. 核心逻辑

### 5.1 主函数实现

In [ ]:
class RatingDogClient:
    """评级狗API客户端"""
    
    def __init__(self):
        self.access_token = None
        self.session = requests.Session()
    
    def login(self):
        """
        登录评级狗API获取访问令牌
        
        Returns:
            bool: 登录是否成功
        """
        try:
            response = self.session.post(
                RATINGDOG_LOGIN_URL,
                headers=get_login_headers(),
                json=get_login_data(),
                timeout=REQUEST_TIMEOUT
            )
            
            result = response.json()
            
            if result.get('success'):
                self.access_token = result['result']['accessToken']
                print("登录成功，已获取访问令牌")
                return True
            else:
                print(f"登录失败: {result.get('error', 'Unknown error')}")
                return False
                
        except Exception as e:
            print(f"登录异常: {e}")
            return False
    
    def fetch_trade_data(self, start_date, end_date, skip_count=0, use_agent_api=False):
        """
        获取成交数据
        
        Args:
            start_date: 开始日期 (YYYY-MM-DD)
            end_date: 结束日期 (YYYY-MM-DD)
            skip_count: 跳过条数（分页）
            use_agent_api: 是否使用中介API
        
        Returns:
            dict: API响应数据
        """
        if not self.access_token:
            if not self.login():
                return None
        
        url = RATINGDOG_AGENT_URL if use_agent_api else RATINGDOG_TRADE_URL
        
        post_data = {
            'BondTypes': [],
            'Citys': [],
            'EndTradedDate': end_date,
            'IssueMethods': [],
            'IssuerTypes': [],
            'MaxResultCount': MAX_RESULT_COUNT,
            'Natures': [],
            'SkipCount': skip_count,
            'SourceTypes': [],
            'StartTradedDate': start_date,
            'TransactionMarkets': [],
            'YYIndustrys': [],
            'YYRatings': [],
        }
        
        for retry in range(MAX_RETRIES):
            try:
                response = self.session.post(
                    url,
                    headers=get_api_headers(self.access_token),
                    json=post_data,
                    timeout=REQUEST_TIMEOUT
                )
                
                result = response.json()
                
                if result.get('success'):
                    return result
                else:
                    print(f"API返回失败，重试 {retry + 1}/{MAX_RETRIES}")
                    sleep(1)
                    
            except Exception as e:
                print(f"请求异常: {e}，重试 {retry + 1}/{MAX_RETRIES}")
                sleep(2)
        
        return None

# 创建客户端实例
client = RatingDogClient()
print("RatingDogClient 客户端创建完成")

### 5.2 辅助函数

In [ ]:
def fetch_all_trade_data(client, start_date, end_date, connection, table_name='cjqb', use_agent_api=False):
    """
    分页获取所有成交数据并存储到数据库
    
    Args:
        client: RatingDogClient实例
        start_date: 开始日期
        end_date: 结束日期
        connection: 数据库连接
        table_name: 目标表名
        use_agent_api: 是否使用中介API
    
    Returns:
        int: 获取的总记录数
    """
    skip_count = 0
    total_count = 0
    fetched_count = 0
    page = 0
    
    print(f"开始获取数据: {start_date} 至 {end_date}")
    
    while True:
        # 获取数据
        result = client.fetch_trade_data(
            start_date, end_date, skip_count, use_agent_api
        )
        
        if result is None:
            print(f"第{page}页数据获取失败")
            break
        
        # 获取总数（第一次）
        if total_count == 0:
            total_count = result.get('result', {}).get('totalCount', 0)
            print(f"总记录数: {total_count}")
        
        # 提取数据
        items = result.get('result', {}).get('items', [])
        
        if not items:
            print("没有更多数据")
            break
        
        # 转换为DataFrame并清洗
        df = pd.DataFrame(items)
        df = clean_trade_data(df)
        
        # 存储到数据库
        try:
            df.to_sql(table_name, connection, if_exists='append', index=False)
            fetched_count += len(df)
            print(f"第{page}页完成: 获取{len(df)}条，累计{fetched_count}条")
        except Exception as e:
            print(f"存储数据失败: {e}")
        
        # 下一页
        skip_count += MAX_RESULT_COUNT
        page += 1
        
        # 添加延迟避免请求过快
        sleep(random.uniform(0.5, 1.5))
    
    print(f"数据获取完成: 共{fetched_count}条")
    return fetched_count

print("辅助函数定义完成")

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
def run_daily_collection(connection, target_table='cjqb', use_agent_api=False):
    """
    执行每日数据采集
    
    Args:
        connection: 数据库连接
        target_table: 目标表名
        use_agent_api: 是否使用中介API
    
    Returns:
        dict: 采集统计信息
    """
    stats = {
        'total_dates': 0,
        'processed_dates': 0,
        'total_records': 0,
        'errors': []
    }
    
    # 获取待采集日期
    pending_dates = get_pending_dates(connection, f'yq.{target_table}')
    
    if len(pending_dates) == 0:
        print("没有待采集的日期")
        return stats
    
    stats['total_dates'] = len(pending_dates)
    
    # 创建客户端
    client = RatingDogClient()
    
    # 遍历日期
    for idx, row in pending_dates.iterrows():
        date_str = row['dt']
        
        try:
            print(f"\n{'='*50}")
            print(f"处理日期: {date_str} ({idx + 1}/{len(pending_dates)})")
            print(f"{'='*50}")
            
            # 获取当天数据
            records = fetch_all_trade_data(
                client, date_str, date_str, connection, target_table, use_agent_api
            )
            
            stats['processed_dates'] += 1
            stats['total_records'] += records
            
        except Exception as e:
            error_msg = f"日期{date_str}处理失败: {e}"
            print(error_msg)
            stats['errors'].append(error_msg)
    
    # 打印统计
    print(f"\n{'='*50}")
    print("采集完成统计:")
    print(f"  总日期数: {stats['total_dates']}")
    print(f"  处理日期数: {stats['processed_dates']}")
    print(f"  总记录数: {stats['total_records']}")
    print(f"  错误数: {len(stats['errors'])}")
    print(f"{'='*50}")
    
    return stats

print("主流程函数定义完成")

In [ ]:
# 测试：获取单日数据（示例）
# 注意：需要配置正确的环境变量才能实际执行

# 检查环境变量是否配置
if RATINGDOG_USERNAME and RATINGDOG_PASSWORD:
    print("环境变量已配置，可以执行数据采集")
    
    # 测试登录
    if client.login():
        print("登录测试成功")
    else:
        print("登录测试失败，请检查用户名和密码")
else:
    print("警告: 请先配置环境变量 RATINGDOG_USERNAME 和 RATINGDOG_PASSWORD")
    print("示例设置方式:")
    print("  export RATINGDOG_USERNAME=your_username")
    print("  export RATINGDOG_PASSWORD=your_password")

### 6.2 结果验证

In [ ]:
def verify_data(connection, table_name='cjqb', days=7):
    """
    验证数据完整性
    
    Args:
        connection: 数据库连接
        table_name: 表名
        days: 检查最近N天
    """
    query = f"""
    SELECT 
        tradedDate,
        COUNT(*) as record_count,
        SUM(tradedAmount) as total_amount
    FROM yq.{table_name}
    WHERE tradedDate >= DATE_SUB(CURDATE(), INTERVAL {days} DAY)
    GROUP BY tradedDate
    ORDER BY tradedDate DESC
    """
    
    try:
        df = pd.read_sql(query, connection)
        print(f"\n最近{days}天数据统计:")
        print(df.to_string())
        return df
    except Exception as e:
        print(f"验证失败: {e}")
        return None

# 执行验证
if db_connection is not None:
    verify_data(db_connection)
else:
    print("数据库未连接，跳过验证")

## 7. 工具函数（可复用）

In [ ]:
# 工具函数集合

def calculate_liquidity_indicators(trade_data):
    """
    计算流动性指标
    
    Args:
        trade_data: 成交数据DataFrame
    
    Returns:
        dict: 流动性指标
    """
    indicators = {}
    
    if trade_data is None or len(trade_data) == 0:
        return indicators
    
    # 成交频率
    if 'tradedDate' in trade_data.columns:
        indicators['trade_frequency'] = len(trade_data['tradedDate'].unique())
    
    # 平均成交金额
    if 'tradedAmount' in trade_data.columns:
        indicators['avg_amount'] = trade_data['tradedAmount'].mean()
        indicators['total_amount'] = trade_data['tradedAmount'].sum()
    
    # 价格波动
    if 'tradedPrice' in trade_data.columns:
        indicators['price_std'] = trade_data['tradedPrice'].std()
        indicators['price_mean'] = trade_data['tradedPrice'].mean()
    
    return indicators


def classify_liquidity(liquidity_indicators):
    """
    流动性分类
    
    Args:
        liquidity_indicators: 流动性指标字典
    
    Returns:
        str: 流动性类别
    """
    thresholds = TRADE_DATA_CONFIG['liquidity_thresholds']
    
    # 简单分类逻辑（可根据实际需求调整）
    trade_freq = liquidity_indicators.get('trade_frequency', 0)
    
    if trade_freq >= 20:
        return '高流动性'
    elif trade_freq >= 10:
        return '中等流动性'
    else:
        return '低流动性'


def export_to_csv(df, filename, output_dir=OUTPUT_DIR):
    """
    导出数据到CSV
    
    Args:
        df: 数据DataFrame
        filename: 文件名
        output_dir: 输出目录
    """
    output_path = output_dir / filename
    df.to_csv(output_path, index=False, encoding='utf-8-sig')
    print(f"数据已导出到: {output_path}")


print("工具函数加载完成")

In [ ]:
# 关闭连接
def cleanup():
    """清理资源"""
    global db_connection, db_engine
    
    if db_connection is not None:
        try:
            db_connection.close()
            print("数据库连接已关闭")
        except:
            pass
    
    if db_engine is not None:
        try:
            db_engine.dispose()
            print("数据库引擎已释放")
        except:
            pass

# 执行清理（可选）
# cleanup()

print("\n重构Notebook 加载完成！")
print("请根据实际需求配置环境变量后执行相关单元格。")